# Phase5
## Model Evaluation
This is the moment of truth. You unlock the test set for the first time - data your model has genuinely never seen and find out if it learned real patterns or just memorized training examples. Everything from Phase1 through Phase4 was preparation for this single honest measurement.

In [ ]:
# Load models and make predictions
# Import
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    average_precision_score,
    f1_score
)

# Load test Data and Trained Models
X_test = pd.read_csv('phase3_output/X_test_final.csv')
y_test = pd.read_csv('phase3_output/y_test_final.csv').squeeze()

lr_model = joblib.load('models/logistic_regression.pkl')
rf_model = joblib.load('models/random_forest_best.pkl')

print(f"Test set: {X_test.shape[0]:,} rows")
print(f"Fraud cases in test set: {y_test.sum()} ({y_test.mean()*100:.4f}%)")
# This is the First time we touch X_test since Phase 2

# Generate Predictions
# Hard predictions: 0 or 1 (used for confusion matrix, classification report)
lr_preds = lr_model.predict(X_test)
rf_preds = rf_model.predict(X_test)

# Soft predictions: probability of fraud 0.0–1.0 (used for ROC, PR curves)
# [:, 1] selects the probability of the positive class (fraud = 1)
lr_probs = lr_model.predict_proba(X_test)[:, 1]
rf_probs = rf_model.predict_proba(X_test)[:, 1]

print("\nPredictions generated for both models.")
print(f"LR flagged {lr_preds.sum()} transactions as fraud")
print(f"RF flagged {rf_preds.sum()} transactions as fraud")
print(f"Actual fraud cases: {y_test.sum()}")


In [ ]:
# The Full Classification Report
def print_report(y_true, y_pred, y_prob, model_name):
    print(f"\n{'n'*56}")
    print(f"  {model_name}")
    print(f"{'='*56}")

    # Classification_report gives precision, recall, f1 per class
    print(classification_report(
        y_true, y_pred,
        target_names=['Normal (0)', 'Fraud (1)'],
        digits=4,          # 4 decimal places for precision
    ))

# ROC-AUC: overall ranking ability
auc = roc_auc_score(y_true, y_prob)
print(f"  ROC-AUC:                {auc:.4f}")

# Average Precision: better than AUC for very imbalanced datasets
# It summarizes the precision-recall curve as a single number
ap = average_precision_score(y_true, y_prob)
print(f"  Average Precision:      {ap:.4f}")

# Extract the raw confusion matrix numbers
tn, fp, fn, tp, = confusion_matrix(y_true, y_pred).ravel()
print(f"\n  Confusion matrix breakdown:")
print(f"    True Negatives  (TN): {tn:>6,} - normal txns correctly passed")
print(f"    False Positives (FP): {fp:>6,} - innocent customers wrongly blocked")
print(f"    False Negatives (FN): {fn:>6,} - FRAUD MISSED (worst outcome)")
print(f"    True Positive   (TP): {tp:>6,} - fraud correctly caught")

# Derived metrics computed manually so you see exactly where they come from
recall    = tp / (tp + fn)
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
fpr       = fp / (fp + tn)      # False Positive Rate

print(f"\n   Recall    = TP/(TP+FN) = {tp}/({tp}+{fn}) = {recall:.4f}")
print(f"   Precision  = TP/(TP+FP) = {tp}/({tp}+{fp}) = {precision:.4f}")
print(f"   F1         = 2xPxR/(P+R)                   = {f1:.4f}")
print(f"   FPR        = FP/(FP+TN) = {fp}/({fp}+{tn} = {fpr:.4f}")

return {'tn':tn, 'fp':fp, 'tp':tp,
        'recall':recall, 'precision':precision,
        'f1':f1, 'auc':auc, 'ap':ap}

lr_metrics = print_report(y_test, lr_preds, lr_probs, 'Logistic Regression')
rf_metrics = print_report(y_test, rf_preds, rf_probs, 'Random Forest')

In [ ]:
# Confusion Matrix Heatmap
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, preds, metrics, name in [
    (axes[0], lr_preds, lr_metrics, 'Logistic Regression'),
    (axes[1], rf_preds, rf_metrics, 'Random Forest')
]:
    cm = confusion_matrix(y_test, preds)

    # Annotate with both count and percentage
    total = cm.sum()
    annot = np.array([
        [f"{cm[0,0]:,}\n({cm[0,0]/total*100:.2f}%)",
         f"{cm[0,1]:,}\n({cm[0,1]/total*100:.2f}%)"],
        [f"{cm[1,0]:,}\n({cm[1,0]/total*100:.2f}%)",
         f"{cm[1,1]:,}\n({cm[1,1]/total*100:.2f}%)"]
    ])

    sns.heatmap(
        cm, ax=ax,
        annot=annot, fmt='',
        cmap='Blues',
        xticklabels=['Normal', 'Fraud'],
        yticklabels=['Normal', 'Fraud'],
        linewidths=0.5,
        cbar=False,
    )
    ax.set_xlabel('Predicted label')
    ax.set_ylabel('True label')
    ax.set_title(
        f'{name}\n'
        f'Recall={metrics["recall"]:.3f}  '
        f'Precision={metrics["precision"]:.3f}  '
        f'F1={metrics["f1"]:.3f}  '
    )

    # Draw a red box highlights False Negatives - fraud your model missed
    # That bottom-left cell is the number you want as close to zero as possible


In [ ]:
# ROC CURVE
plt.figure(figsize=(8, 7))

for probs, name, color in [
    (lr_probs, 'Logistic Regression', '#D85A30'),
    (rf_probs, 'Random Forest',       '#185FA5'),
]:
    fpr_vals, tpr_vals, thresholds = roc_curve(y_test, probs)
    auc_val = roc_auc_score(y_test, probs)

    plt.plot(fpr_vals, tpr_vals,
             label=f'{name}  (AUC = {auc_val:.4f})',
             linewidth=2.5, color=color)

# Shade the area under Random Forest curve
rf_fpr, rf_tpr, _ = roc_curve(y_test, probs)
plt.fill_between(rf_fpr, rf_tpr, alpha=0.08, color='#185FA5')

# Baseline: a random model follows the diagonal
plt.scatter([0], [1], color='#1D9E75', s=80, zorder=5, label='Perfect model')

plt.xlabel('False Positive Rate  (innocent customers blocked)')
plt.ylabel('True Positive Rate   (fraud caught = Recall')
plt.title('ROC curve - how well each model ranks fraud above normal')
plt.legend(loc='lower right')
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

# HOW TO READ THIS:
# Every point on the curve = one threshold setting
# Top-left corner = perfect: catch all fraud, block nobody innocent
# The closer the curve hugs that corner, the better
# AUC = 0.98 means the model ranks a random fraud txn above a
#       random normal txn 98% of the time



In [ ]:
#  PRECISION-RECALL CURVE
# ROC curves can look optimistic on imbalanced data because TN dominates
# PR curves focus only on the positive (fraud) class — no TN involved
# This gives a more honest picture when positives are rare

plt.figure(figsize=(8, 7))

for probs, name, color in [
    (lr_probs, 'Logistic Regression', '#D85A30'),
    (rf_probs, 'Random Forest',       '#185FA5')
]:
    precision_vals, recall_vals, _ = precision_recall_curve(y_test, probs)
    ap_val = average_precision_score(y_test, probs)

    plt.plot(recall_vals, precision_vals,
             label=f'{name}  (AP = {ap_val:.4f})',
             linewidth=2.5, color=color)

# Baseline: a random classifier sits at the fraud prevalence rate
baseline = y_test.mean()
plt.axhline(y=baseline, color='gray', linestyle='--', linewidth=1,
            label=f'Random baseline (AP = {baseline:.4f})')
plt.xlabel('Recall (fraction of fraud cases caught)')
plt.ylabel('Precision  (fraction of fraud alerts that are real)')
plt.title('Precision-Recall Curve - fraud detection')
plt.grid(alpha=0.25)
plt.tight_layout()
ply.show()


# HOW TO READ THIS:
# Top-right corner = perfect: catch all fraud AND every alert is correct
# The "elbow" where precision drops sharply = practical operating threshold
# AP (Average Precision) = area under this curve
# Our baseline is tiny (~0.0017) so any AP >> 0.0017 is meaningful progress


In [ ]:
# Threshold Tuning
def find_best_threshold(y_true, y_prob, model_name):
    """
    Sweep every threshold and compute metrics.
    Returns a DataFrame and highlights the best operating point
    """
    thresholds = np.arange(0.05, 0.95, 0.05)
    results = []

    for t in thresholds:
        preds = (y_prob >= t).astype(int)
        cm    = confusion_matrix(y_true, preds)
        tn, fp, fn, tp = cm.ravel()

        recall    = tp / (tp + fn)    if (tp + fn) > 0 else 0
        precison  = tp / (tp + fp)    if (tp + fp) > 0 else 0
        f1        = (2 * precision * recall / (precision + recall)
                    if (precision + recall) > 0 else 0)
        fpr       = fp / (fp + tn)    if (fp + tn) > 0 else 0

        results.append({
            'threshold':  round(float(t), 2),
            'recall':     round(recall,   3),
            'precision':  round(precison,  3),
            'f1':         round(f1,        3),
            'fn_count':   int(fn),
            'fp_count':   int(fp),
            'fpr':        round(fpr,      4)
        })

    df = pd.DataFrame(results)
    print(f"\n{model_name} - threshold sweep:")
    print(df.to_string(index=False))

    # Strategy 1: maximize F1 (balance trade-off)
    best_f1_row = df.loc[df['f1'].idxmax()]

    # Strategy 2: maximize recall while keeping precision above 50 %
    high_recall = df[df['precision'] >= 0.50]
    best_recall_row = (high_recall.loc[high_recall['recall'].idxmax()]
                        if not high_recall.empty else best_f1_row)

    print(f"\n  Best threshold for F1:          "
          f"{best_f1_row['threshold']}  "
          f"(Recall={best_f1_row['recall']:.3f}, "
          f"Precision={best_f1_row['precision']:.3f}, "
          f"F1={best_f1_row['f1']:.3f})"
          )

    print(f"  Best threshold for recall (P>=0.5):  "
          f"{best_recall_row['threshold']}   "
          f"(Recall={best_recall_row['recall']:.3f}, "
          f"Precision={best_recall_row['precision']:.3f}, "
          f"FN={best_recall_row['fn_count']} missed frauds)"
          )
    return df

lr_thresh_df = find_best_threshold(y_test, lr_probs, 'Logistic Regression')
rf_thresh_df = find_best_threshold(y_test, rf_probs, 'Random Forest')

# Visualize the threshold Trade-off
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, df, name in[
    (axes[0], lr_thresh_df, 'Logistic Regression'),
    (axes[1], rf_thresh_df, 'Random Forest')
]:
    ax.plot(df['threshold'], df['recall'],   'o-',
            color='#E24B4A', label='Recall',  lw=2)
    ax.plot(df['threshold'], df['precision'], 's-',
            color='#378ADD', label='Precision', lw=2)
    ax.plot(df['threshold'], df['f1'],       '^-',
            color='#1D9E75', label='F1 Score',  lw=2)

    # Mark default threshold
    ax.axvline(x=0.5, color='gray', linestyle='--',
               linewidth=1, label='Default (0.5)')

    ax.set_xlabel('Decision threshold')
    ax.set_ylabel('Score')
    ax.set_title(f'{name} - threshold trade-off')
    ax.legend()
    ax.grid(alpha=0.25)
    ax.set_ylim(0, 1.05)

plt.suptitle('Precision vs Recall at every threshold', y=1.02)
plt.tight_layout()
plt.show()


# Lower threshold → higher recall (catch more fraud)
#                 → lower precision (more false alarms)
# Choosing the threshold is a business decision, not a technical one.
# Ask: "Is missing 1 fraud worse than blocking 10 innocent customers?"
# For most banks: yes — lower the threshold.


In [ ]:
#  APPLY THE CHOSEN THRESHOLD
# After inspecting the threshold sweep, choose your operating threshold
# Here we choose the one that maximizes recall while keeping precision >= 50%

CHOSEN_THRESHOLD = 0.30  # Adjust based on your threshold sweep output

rf_preds_tuned = (rf_prbs >= CHOSEN_THRESHOLD).astype(int)

print(f"Final evaluation - Random Forest at threshold = "{CHOSEN_THRESHOLD}\n")

tn, fp, fn, tp = confusion_matrix(y_test, rf_preds_tuned).ravel()

recall = tp / (tp + fn)
prcision = tp / (tp + fp) if (tp + fp) > 0 else 0
f1    = 2 * precision * recall / (precision + recall)

print(f"  Fraud cases in test set:     {y_test.sum()}")
print(f"  Fraud caught (TP):           {tp}  ({recall*100:.1f}% of all fraud)")
print(f"  Fraud missed (FN):           {fn}  ({fn/(tp+fn)*100:.1f}% slipped through)")
print(f"  Innocent blocked (FP):       {fp}")
print(f"  Innocent passed (TN):        {tn:,}")
print(f"\n  Recall:    {recall:.4f}")
print(f"  Precision: {precision:.4f}")
print(f"  F1 Score:  {f1:.4f}")


In [ ]:
#  FINAL MODEL COMPARISON TABLE
comparison = pd.DataFrame({
    'Metric': [
        'Recall (fraud caught)',
        'Precision (alert accuracy)',
        'F1 Score',
        'ROC-AUC',
        'Average Precision',
        'Fraud missed (FN)',
        'False alarms (FP)'
    ],
    'Logistic Regression': [
        f"{lr_metrics['recall']:.4f}",
        f"{lr_metrics['precision']:.4f}",
        f"{lr_metrics['f1']:.4f}",
        f"{lr_metrics['auc']:.4f}",
        f"{lr_metrics['ap']:.4f}",
        f"{lr_metrics['fn']}",
        f"{lr_metrics['fp']:,}"
    ],
    'Random Forest': [
        f"{rf_metrics['recall']:.4f}",
        f"{rf_metrics['precision']:.4f}",
        f"{rf_metrics['f1']:.4f}",
        f"{rf_metrics['auc']:.4f}",
        f"{rf_metrics['ap']:.4f}",
        f"{rf_metrics['fn']}",
        f"{rf_metrics['fp']:,}"
    ]
})

print(comparison.to_string(index=False))

# Which to choose?
# If Random Forest recall >> LR recall → use Random Forest
# If they're nearly equal but LR is faster → use LR (simpler = better)
# For fraud: always prioritise the model with higher recall
winner = ('Random Forest' if rf_metrics['recall'] > lr_metrics['recall']
          else 'Logistic Regression')
print(f"\nRecommended model: {winner}")
print(f"Reason: higher recall = fewer fraud cases missed")